# Week #2(b): Image Grid Transformations

[Open this notebook in Colab](https://colab.research.google.com/github/mmoffitt/engineering-intelligence/blob/main/notebooks/week02b.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 2(a), you practiced functions, loops, and lists of lists. Now you will use those skills to solve a series of image-grid puzzles.

By the end of this notebook, you will be able to:
- read a grid as rows and columns
- describe a transformation in plain language
- write functions that inspect and transform grids
- combine small functions to solve a complete grid puzzle

The goal is not to guess quickly. The goal is to **observe, describe, implement, and test**.

## A problem-solving routine for grid puzzles

Each puzzle gives you small input and output grids. Each number represents a color. Your job is to infer the rule that changes the input into the output.

Use this routine:

1. **Observe:** What stayed the same? What changed?
2. **Describe:** State the rule in one clear sentence.
3. **Implement:** Turn that sentence into a Python function.
4. **Test:** Try the function on every example, including a new grid.

Do not begin coding until you can describe a possible rule. A clear sentence is often the first version of your algorithm.

## Setup

Run the next cell once. It provides:
- `show_grid(...)` for drawing image grids
- `show_pair(...)` for placing an input beside its output
- the **Check Exercise** buttons used later

In our color system, `0` is black, `1` is blue, `2` is red, and so on.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)
    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(3, 3))
    draw_grid(ax, grid, title)
    plt.show()


def show_pair(input_grid, output_grid):
    _, axes = plt.subplots(1, 2, figsize=(6, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    plt.show()


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            check_fn()

    button.on_click(on_click)
    display(button, output)


print("Setup complete.")

## Reading a grid

A grid is a **list of rows**, and each row is a list of integers.

```python
grid = [
    [0, 0, 2],
    [0, 2, 0],
    [2, 0, 0]
]
```

Python counts from zero:
- `grid[0]` is the first row: `[0, 0, 2]`
- `grid[1][1]` is the cell in row 1, column 1: `2`
- `len(grid)` is the number of rows
- `len(grid[0])` is the number of columns

Run the next cell and compare the Python list with the picture.

In [ ]:
example_grid = [
    [0, 0, 2],
    [0, 2, 0],
    [2, 0, 0]
]

print("First row:", example_grid[0])
print("Center cell:", example_grid[1][1])
show_grid(example_grid, "A 3 × 3 grid")

## Exercise 1: Find a grid's size

Write `grid_size(grid)` so it returns the number of rows and columns as a tuple.

Examples:
- `grid_size([[1, 2], [3, 4]])` should return `(2, 2)`
- `grid_size([[1, 2, 3]])` should return `(1, 3)`

You may assume that the grid has at least one row and one column.

**Hint:** Use `len` twice.

In [ ]:
# Write your solution for Exercise 1 here

def grid_size(grid):
    return None

In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [
        ([[1, 2], [3, 4]], (2, 2)),
        ([[1, 2, 3]], (1, 3)),
        ([[0], [0], [0]], (3, 1)),
    ]
    try:
        for grid, expected in tests:
            got = grid_size(grid)
            if got != expected:
                print(f"Close: grid_size({grid}) returned {got}, expected {expected}.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Great job. Exercise 1 is correct.")


make_check_button(check_exercise_1)

## Exercise 2: Count a color

Grid-puzzle rules often depend on how many cells have a certain color.

Write `count_color(grid, color)` so it returns the number of cells equal to `color`.

Example:

```python
grid = [
    [0, 2, 0],
    [2, 2, 0]
]
```

`count_color(grid, 2)` should return `3`.

**Hint:** Use one loop for rows and a second loop for cells.

In [ ]:
# Write your solution for Exercise 2 here

def count_color(grid, color):
    return None

In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    tests = [
        (([[0, 2, 0], [2, 2, 0]], 2), 3),
        (([[1, 1], [1, 0]], 1), 3),
        (([[4, 4, 4]], 0), 0),
        (([[0]], 0), 1),
    ]
    try:
        for (grid, color), expected in tests:
            got = count_color(grid, color)
            if got != expected:
                print(f"Close: counting color {color} returned {got}, expected {expected}.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Nice work. Exercise 2 is correct.")


make_check_button(check_exercise_2)

## Exercise 3: Replace one color

Study this example before coding.

In [ ]:
replace_example_input = [
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
]
replace_example_output = [
    [0, 2, 0],
    [2, 0, 2],
    [0, 2, 0]
]

show_pair(replace_example_input, replace_example_output)

What changed? Every blue cell (`1`) became red (`2`). The black background (`0`) stayed unchanged.

Write `replace_color(grid, old_color, new_color)` so it returns a **new grid** with every `old_color` replaced by `new_color`.

Do not change the original grid. Build a new list one row at a time.

A useful pattern is:

```python
new_grid = []
for row in grid:
    new_row = []
    # inspect each cell and append a value to new_row
    new_grid.append(new_row)
return new_grid
```

In [ ]:
# Write your solution for Exercise 3 here

def replace_color(grid, old_color, new_color):
    return None

In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    tests = [
        (
            [[0, 1, 0], [1, 0, 1]], 1, 2,
            [[0, 2, 0], [2, 0, 2]],
        ),
        (
            [[3, 3], [0, 3]], 3, 4,
            [[4, 4], [0, 4]],
        ),
        (
            [[0, 0]], 7, 8,
            [[0, 0]],
        ),
    ]
    try:
        for grid, old_color, new_color, expected in tests:
            original = copy.deepcopy(grid)
            got = replace_color(grid, old_color, new_color)
            if got != expected:
                print(f"Close: got {got}, expected {expected}.")
                return
            if grid != original:
                print("The transformed grid is right, but the original grid was changed. Build a new grid instead.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Excellent. Exercise 3 is correct and preserves its input.")


make_check_button(check_exercise_3)

## Exercise 4: Flip a grid horizontally

A horizontal flip acts like a mirror placed beside the grid: each row appears in reverse order.

```text
[1, 0, 0]        [0, 0, 1]
[0, 2, 0]   →    [0, 2, 0]
[3, 3, 0]        [0, 3, 3]
```

Write `flip_horizontal(grid)` so it returns a new, horizontally flipped grid.

Python can reverse a list with `row[::-1]`. Use that once for each row. Again, do not change the original grid.

In [ ]:
# Run this example, then write your function below.
flip_example_input = [
    [1, 0, 0],
    [0, 2, 0],
    [3, 3, 0]
]
flip_example_output = [
    [0, 0, 1],
    [0, 2, 0],
    [0, 3, 3]
]
show_pair(flip_example_input, flip_example_output)


def flip_horizontal(grid):
    return None

In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    tests = [
        (
            [[1, 0, 0], [0, 2, 0], [3, 3, 0]],
            [[0, 0, 1], [0, 2, 0], [0, 3, 3]],
        ),
        (
            [[1, 2, 3, 4]],
            [[4, 3, 2, 1]],
        ),
        (
            [[1], [2]],
            [[1], [2]],
        ),
    ]
    try:
        for grid, expected in tests:
            original = copy.deepcopy(grid)
            got = flip_horizontal(grid)
            if got != expected:
                print(f"Close: got {got}, expected {expected}.")
                return
            if grid != original:
                print("The result is right, but the original grid was changed. Return a new grid.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Fantastic. Exercise 4 is correct.")


make_check_button(check_exercise_4)

## Final challenge: Solve a complete grid puzzle

Now infer a rule from two training examples.

Look carefully before reading further. What changes about the colors? What changes about the positions?

In [ ]:
train_1_input = [
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
]
train_1_output = [
    [0, 0, 2],
    [0, 2, 0],
    [2, 0, 0]
]

train_2_input = [
    [0, 1, 0, 0],
    [1, 1, 0, 0]
]
train_2_output = [
    [0, 0, 2, 0],
    [0, 0, 2, 2]
]

print("Training example 1")
show_pair(train_1_input, train_1_output)
print("Training example 2")
show_pair(train_2_input, train_2_output)

Write your rule in one sentence before coding:

> The output is the input ________________________________.

Then write `solve_task(grid)`.

**Hint 1:** This task combines the transformations from Exercises 3 and 4.

**Hint 2:** Small functions can call other functions. For example:

```python
def larger_job(value):
    first_result = small_job_one(value)
    final_result = small_job_two(first_result)
    return final_result
```

Your function should work for grids of different sizes, not only the two training examples.

In [ ]:
# Write your solution for the final challenge here

def solve_task(grid):
    return None

In [ ]:
#@title Check Final Challenge

def check_final_challenge():
    tests = [
        (train_1_input, train_1_output),
        (train_2_input, train_2_output),
        (
            [[1, 0, 1], [0, 0, 1]],
            [[2, 0, 2], [2, 0, 0]],
        ),
        (
            [[0, 1, 0, 1, 1]],
            [[2, 2, 0, 2, 0]],
        ),
    ]
    try:
        for grid, expected in tests:
            original = copy.deepcopy(grid)
            got = solve_task(grid)
            if got != expected:
                print("Not yet. Recheck both the positions and the colors.")
                print("Input:   ", grid)
                print("Got:     ", got)
                print("Expected:", expected)
                return
            if grid != original:
                print("The output is correct, but the input changed. Preserve the original grid.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("You solved the grid puzzle. Excellent work!")


make_check_button(check_final_challenge)

## Apply your rule to a test grid

The final puzzle includes a test input whose output is hidden. Predict its output on paper first. Then run your solver and display the result.

In [ ]:
test_input = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [1, 1, 0, 0]
]

# Finish solve_task before running this cell.
test_output = solve_task(test_input)
if test_output is None:
    print("Finish solve_task(grid), then run this cell again.")
else:
    show_pair(test_input, test_output)

## Stretch challenges

Complete these only after all five checks pass. Try them in order.

1. Write `flip_vertical(grid)` to reverse the order of the rows.
2. Write `transpose_grid(grid)` to turn rows into columns.
3. Write `most_common_color(grid)` to return the color that occurs most often.
4. Make your own input/output pair using `show_pair`. Write the rule in one sentence and ask someone else to infer it.

For each function, invent at least two tests before you consider it finished.

## Wrap-up and reflection

You practiced a reusable problem-solving workflow:

1. inspect examples
2. propose a rule
3. implement the rule
4. test it on multiple grids

Before sharing your notebook, answer these questions in a text cell below:

- What was the final task's rule, in one sentence?
- Which exercise required the most debugging?
- What did you check when your output was wrong?
- What grid transformation would you like to learn next?

### My reflection

- **The final task's rule:**
- **The exercise that required the most debugging:**
- **What I checked when my output was wrong:**
- **A transformation I want to learn next:**

## Share your completed notebook

When you finish:

1. Confirm that all five **Check Exercise** buttons report success.
2. In Colab, click **Share**.
3. Add **michael.d.moffitt@gmail.com**.
4. Set permission to **Viewer** or **Commenter**.
5. Send the share invite.

Include any questions that came up while you worked. Getting stuck, testing an idea, and revising it are all normal parts of engineering.